# Frozen END-direction evaluation on SST, IMDb, and DynaSent

This notebook reuses the **END-position** directions and layers selected on ToyMovieReview **ADVERB logit-flip percent** in the completed sentiment-position run `2026-09-18_20-22_CDT`. The current resume configuration evaluates the frozen mean-difference, logistic-regression, and one-dimensional DAS directions on SST, IMDb, DynaSent R1, and DynaSent R2 for Qwen3-0.6B Base only, while preserving the completed GPT-2 Small result files.

No direction is refit and no layer is reselected. The four datasets in this notebook are evaluation-only. All CSVs are written to a new minute-stamped Google Drive run using the current America/Chicago date.

## Before running

1. Use a Colab GPU runtime.
2. Confirm the completed parent run exists under `MyDrive/sentiment-geometry/sentiment-position-comparison/runs`.
3. When prompted, provide a Hugging Face token that can read the private SST, IMDb, and DynaSent repositories. The token is hidden, passed only through the configured environment variable during evaluation, and removed afterward. The final cell verifies that the environment variable and notebook secret cache are empty.
4. Leave dataset revisions pinned for a reportable run.
5. `RESUME_RUN_ID` is pinned to the interrupted `2026-09-22_00-18_CDT` run. The evaluator requires and reuses its completed GPT-2 result files, then runs Qwen only.

In [1]:
# --------------------------- User settings ---------------------------
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag. Existing checkouts must match it.

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
SOURCE_RUN_ID = "2026-09-18_20-22_CDT"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = "2026-09-22_00-18_CDT"  # Reuse completed GPT-2 results.

DEVICE = "cuda"
DTYPE = "auto"
ALL_MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]
EVALUATION_MODEL_NAMES = ["qwen-0.6b"]
REUSE_COMPLETED_MODEL_NAMES = ["gpt2-small"]
MODEL_BATCH_SIZES = {
    "qwen-0.6b": 8,
}
RUN_EVALUATION = True

SST_REVISION = "bcffb933a34a48b409a7caf3a53f0fe7bb8152cc"
IMDB_REVISION = "06586d20342aa46fa61525b1f7609ab065983e3e"
DYNASENT_REVISION = "f4aa6cc90bfeda2080e6445592a8f689c974fd3d"

## 1. Install and verify the project

The notebook imports the reusable experiment API; fitting, dataset loading, directional patching, metrics, and persistence remain in `sentiment_geometry`.

In [2]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)

project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )

module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(
        sentiment_geometry.__path__, prefix="sentiment_geometry."
    )
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)

required_apis = {
    "sentiment_geometry.experiments": {
        "FrozenDirectionEvaluationConfig",
        "FrozenEvaluationDataset",
        "load_frozen_direction_selections",
        "run_frozen_sentiment_direction_evaluation",
    },
    "sentiment_geometry.persistence": {
        "RunArtifactStore",
        "maybe_mount_google_drive",
        "prepare_timestamped_run",
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(f"{module_name} is missing {missing_apis}.")

print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)
print(f"Successfully imported {len(module_names) + 1} sentiment_geometry modules.")

Project commit: 2d2da9eafdeb423468bba98058f1bd7594f31677
Imported package from: /content/sentiment-manifold/sentiment_geometry
Successfully imported 61 sentiment_geometry modules.


## 2. Mount Drive and create today's evaluation run

The parent run is opened read-only. The new run receives its own date-and-time directory.

In [3]:
import json

import torch

from sentiment_geometry.persistence import maybe_mount_google_drive, prepare_timestamped_run

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")

maybe_mount_google_drive(True)
SOURCE_RUN_ROOT = (
    Path(DRIVE_STORAGE_ROOT)
    / "sentiment-position-comparison"
    / "runs"
    / SOURCE_RUN_ID
)
SOURCE_MANIFEST_PATH = SOURCE_RUN_ROOT / "run_manifest.json"
if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))
if source_manifest.get("status") != "completed":
    raise RuntimeError(
        f"Source run {SOURCE_RUN_ID} is not completed: {source_manifest.get('status')!r}"
    )

if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name="end-position-ood-evaluation",
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )

print("Parent run:    ", SOURCE_RUN_ROOT)
print("New run ID:    ", RUN_LAYOUT.run_id)
print("New run root:  ", RUN_LAYOUT.root)
print("Result tables: ", RUN_LAYOUT.results_dir)
if torch.cuda.is_available():
    print("GPU:           ", torch.cuda.get_device_name(0))

Mounted at /content/drive
Parent run:     /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-18_20-22_CDT
New run ID:     2026-09-22_00-18_CDT
New run root:   /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT
Result tables:  /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/results
GPU:            NVIDIA A100-SXM4-40GB


## 3. Authenticate to the private Hugging Face datasets

The token is never printed, written to Drive, placed in a command argument, or retained after evaluation.

In [4]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): " ).strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def delete_runtime_secret(name):
    value = _RUNTIME_SECRETS.pop(name, None)
    if value is not None:
        del value

def clear_hf_credentials(env_name="HF_TOKEN"):
    os.environ.pop(env_name, None)
    delete_runtime_secret("HF_TOKEN")
    reset_sessions()
    gc.collect()

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
except BaseException:
    clear_hf_credentials()
    raise
finally:
    del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

Authenticated to Hugging Face as kokolamba. Token value was not displayed.


## 4. Configure frozen END-position evaluation

The source run's `layer_selection.csv` chooses exactly one layer per model and method using ToyMovieReview ADVERB logit-flip percent. The corresponding saved direction checkpoint is reused.

In [ ]:
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from sentiment_geometry.experiments import (
    FrozenDirectionEvaluationConfig,
    FrozenEvaluationDataset,
    SentimentPositionExperimentConfig,
)
from sentiment_geometry.persistence import RunArtifactStore

base_config = SentimentPositionExperimentConfig.load(
    PROJECT_ROOT / "configs/sentiment_position_comparison.yaml"
)
if [model.name for model in base_config.models] != ALL_MODEL_NAMES:
    raise RuntimeError("The pinned model list changed unexpectedly.")
if set(EVALUATION_MODEL_NAMES) & set(REUSE_COMPLETED_MODEL_NAMES):
    raise RuntimeError("A model cannot be both evaluated and reused.")
if set(EVALUATION_MODEL_NAMES) | set(REUSE_COMPLETED_MODEL_NAMES) != set(ALL_MODEL_NAMES):
    raise RuntimeError("Every model must be either evaluated or reused.")
if set(MODEL_BATCH_SIZES) != set(EVALUATION_MODEL_NAMES):
    raise RuntimeError("MODEL_BATCH_SIZES must specify only the evaluated models.")
model_configs = {model.name: model for model in base_config.models}
evaluation_models = [model_configs[name] for name in EVALUATION_MODEL_NAMES]
for model in evaluation_models:
    model.device = DEVICE
    model.dtype = DTYPE
    model.batch_size = MODEL_BATCH_SIZES[model.name]

METHOD_ORDER = ["mean_diff", "logistic_regression", "das"]
DATASET_ORDER = ["sst", "imdb", "dynasent_r1", "dynasent_r2"]
MODEL_CONFIG_SLUGS = {
    "gpt2-small": "gpt2_small",
    "qwen-0.6b": "qwen_0_6b",
}

datasets = [
    FrozenEvaluationDataset(
        name="sst",
        repo_id="kokolamba/sentiment-manifold-sst-pythia-2.8b",
        revision=SST_REVISION,
        configs={
            model: f"tigges_{slug}_directed_pairs"
            for model, slug in MODEL_CONFIG_SLUGS.items()
        },
    ),
    FrozenEvaluationDataset(
        name="imdb",
        repo_id="kokolamba/sentiment-manifold-imdb-pythia-2.8b",
        revision=IMDB_REVISION,
        configs={
            model: f"{slug}_directed_pairs"
            for model, slug in MODEL_CONFIG_SLUGS.items()
        },
    ),
    FrozenEvaluationDataset(
        name="dynasent_r1",
        repo_id="kokolamba/sentiment-manifold-dynasent-r1-r2-pythia-2.8b",
        revision=DYNASENT_REVISION,
        configs={
            model: f"r1_{slug}_directed_pairs"
            for model, slug in MODEL_CONFIG_SLUGS.items()
        },
    ),
    FrozenEvaluationDataset(
        name="dynasent_r2",
        repo_id="kokolamba/sentiment-manifold-dynasent-r1-r2-pythia-2.8b",
        revision=DYNASENT_REVISION,
        configs={
            model: f"r2_{slug}_directed_pairs"
            for model, slug in MODEL_CONFIG_SLUGS.items()
        },
    ),
]

config = FrozenDirectionEvaluationConfig(
    source_run_root=str(SOURCE_RUN_ROOT),
    output_dir=str(RUN_LAYOUT.results_dir),
    models=evaluation_models,
    datasets=datasets,
    methods=METHOD_ORDER,
    fit_position="final",
    selection_dataset="toy_adverbs",
    selection_metric="logit_flip_percent",
    hf_token_env=base_config.data.hf_token_env,
    reuse_completed_models=REUSE_COMPLETED_MODEL_NAMES,
)
config.validate()

RunArtifactStore(RUN_LAYOUT.root).write_json(
    "requested_config.json", config.to_dict()
)
RUN_LAYOUT.update_manifest(
    status="configured",
    metadata={
        "project_commit": project_commit,
        "source_run_id": SOURCE_RUN_ID,
        "source_run_root": str(SOURCE_RUN_ROOT),
        "models": [asdict(model) for model in base_config.models],
        "evaluated_models": EVALUATION_MODEL_NAMES,
        "reused_completed_models": REUSE_COMPLETED_MODEL_NAMES,
        "methods": METHOD_ORDER,
        "fit_position": "final",
        "selection_dataset": "toy_adverbs",
        "selection_metric": "logit_flip_percent",
        "evaluation_datasets": DATASET_ORDER,
        "configuration": "requested_config.json",
    },
)

display(pd.DataFrame([
    {
        "dataset": dataset.name,
        "repository": dataset.repo_id,
        "revision": dataset.revision,
        "split": dataset.split,
    }
    for dataset in datasets
]))

,dataset,repository,revision,split
0,sst,kokolamba/sentiment-manifold-sst-pythia-2.8b,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,test
1,imdb,kokolamba/sentiment-manifold-imdb-pythia-2.8b,06586d20342aa46fa61525b1f7609ab065983e3e,test
2,dynasent_r1,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,test
3,dynasent_r2,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,test


## 5. Display and validate the frozen ADVERB-best layers

This table is exploratory reporting only. It is loaded from the parent run's saved CSVs and does not perform a new search. Each cell is `Lxx (ADVERB logit-flip %)`.

In [6]:
from sentiment_geometry.experiments import load_frozen_direction_selections

MODEL_LABELS = {
    "gpt2-small": "GPT-2 Small",
    "qwen-0.6b": "Qwen3-0.6B Base",
}
METHOD_LABELS = {
    "mean_diff": "Mean Difference",
    "logistic_regression": "Logistic Regression",
    "das": "DAS (1D)",
}

selection_rows = []
for model in base_config.models:
    for selected in load_frozen_direction_selections(config, model):
        selection_rows.append({
            "model": selected.model,
            "method": selected.method,
            "fit_position": selected.fit_position,
            "selected_layer": selected.selected_layer,
            "selection_dataset": selected.selection_dataset,
            "selection_metric": selected.selection_metric,
            "selection_value_percent": selected.selection_value_percent,
            "source_direction_checkpoint": str(selected.checkpoint_path),
        })
FROZEN_SELECTION = pd.DataFrame(selection_rows)
if len(FROZEN_SELECTION) != len(ALL_MODEL_NAMES) * len(METHOD_ORDER):
    raise RuntimeError("The frozen END-position selection grid is incomplete.")
if set(FROZEN_SELECTION["selection_dataset"]) != {"toy_adverbs"}:
    raise RuntimeError("A layer was not selected on ToyMovieReview ADVERB.")
if set(FROZEN_SELECTION["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("A layer was not selected by ADVERB logit-flip percent.")
if set(FROZEN_SELECTION["fit_position"]) != {"final"}:
    raise RuntimeError("A selected direction was not fitted at END.")

layer_display = FROZEN_SELECTION.copy()
layer_display["layer_and_score"] = layer_display.apply(
    lambda row: (
        f"L{int(row['selected_layer']):02d} "
        f"({float(row['selection_value_percent']):.1f}%)"
    ),
    axis=1,
)
best_layer_table = layer_display.pivot(
    index="method", columns="model", values="layer_and_score"
).reindex(index=METHOD_ORDER, columns=ALL_MODEL_NAMES)
best_layer_table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
best_layer_table.columns = [MODEL_LABELS[model] for model in ALL_MODEL_NAMES]
display(best_layer_table.style.set_properties(**{"text-align": "center"}))

,GPT-2 Small,Qwen3-0.6B Base
Mean Difference,L10 (20.6%),L23 (36.7%)
Logistic Regression,L10 (20.6%),L23 (36.7%)
DAS (1D),L10 (61.8%),L26 (83.3%)


## 6. Run the frozen OOD evaluations

The Hugging Face token is present only while the private directed-pair shards are resolved and loaded. The package writes aggregate metrics, per-case patching records, selected-layer provenance, dataset summaries, direction metadata, and resolved configuration CSV/JSON files directly to the new Drive run.

In [7]:
from sentiment_geometry.experiments import run_frozen_sentiment_direction_evaluation

if not RUN_EVALUATION:
    print("Evaluation execution is disabled. Set RUN_EVALUATION = True.")
else:
    os.environ.pop(config.hf_token_env, None)
    try:
        os.environ[config.hf_token_env] = get_runtime_secret("HF_TOKEN")
        RUN_LAYOUT.update_manifest(status="running")
        completed_results_dir = run_frozen_sentiment_direction_evaluation(config)
        if completed_results_dir.resolve() != RUN_LAYOUT.results_dir.resolve():
            raise RuntimeError(f"Unexpected results directory: {completed_results_dir}")
        RUN_LAYOUT.update_manifest(status="evaluation-completed")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        clear_hf_credentials(config.hf_token_env)

    print("Evaluation results saved to:", completed_results_dir)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

r2_qwen_0_6b_directed_pairs/test-00000-o(…):   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Evaluation results saved to: /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/results


## 7. Audit the saved evaluation tables

The audit checks the complete model × method × dataset grid, uniqueness, frozen-layer reuse, per-model case records, and requested/resolved dataset revisions before marking the run completed.

In [8]:
from datetime import datetime
from itertools import product
from zoneinfo import ZoneInfo

METRICS_PATH = RUN_LAYOUT.results_dir / "all_models_metrics.csv"
SELECTION_PATH = RUN_LAYOUT.results_dir / "all_models_layer_selection.csv"
SUMMARY_PATH = RUN_LAYOUT.results_dir / "all_models_dataset_summary.csv"
for required_path in (METRICS_PATH, SELECTION_PATH, SUMMARY_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

metrics = pd.read_csv(METRICS_PATH)
saved_selection = pd.read_csv(SELECTION_PATH)
dataset_summary = pd.read_csv(SUMMARY_PATH)
metric_keys = ["model", "method", "fit_position", "dataset"]
if metrics.duplicated(metric_keys).any():
    raise RuntimeError("Duplicate aggregate metric cells were saved.")
expected_cells = set(product(ALL_MODEL_NAMES, METHOD_ORDER, ["final"], DATASET_ORDER))
actual_cells = set(metrics[metric_keys].itertuples(index=False, name=None))
if expected_cells != actual_cells:
    raise RuntimeError(
        f"Incomplete metric grid. Missing={sorted(expected_cells - actual_cells)}; "
        f"unexpected={sorted(actual_cells - expected_cells)}"
    )
if len(saved_selection) != len(ALL_MODEL_NAMES) * len(METHOD_ORDER):
    raise RuntimeError("Saved frozen-layer selection grid is incomplete.")
if set(saved_selection["selection_dataset"]) != {"toy_adverbs"}:
    raise RuntimeError("Saved selection provenance is not exclusively ADVERB.")
if set(saved_selection["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("Saved selection metric is not logit-flip percent.")

for model_name in ALL_MODEL_NAMES:
    model_dir = RUN_LAYOUT.results_dir / model_name
    for filename in (
        "metrics.csv",
        "patching_records.csv",
        "layer_selection.csv",
        "dataset_summary.csv",
        "direction_metadata.csv",
        "resolved_config.json",
    ):
        if not (model_dir / filename).is_file():
            raise FileNotFoundError(model_dir / filename)
    patching_records = pd.read_csv(model_dir / "patching_records.csv")
    expected_records = int(
        dataset_summary.loc[dataset_summary["model"] == model_name, "n_directed_cases"].sum()
    ) * len(METHOD_ORDER)
    if len(patching_records) != expected_records:
        raise RuntimeError(
            f"{model_name} saved {len(patching_records)} per-case records; "
            f"expected {expected_records}."
        )

completed_at = datetime.now(ZoneInfo(RUN_LAYOUT.timezone_name)).isoformat(timespec="minutes")
RUN_LAYOUT.update_manifest(
    status="completed",
    metadata={
        "completed_at": completed_at,
        "aggregate_metric_rows": len(metrics),
        "frozen_layer_rows": len(saved_selection),
        "dataset_summary_rows": len(dataset_summary),
    },
)
display(dataset_summary.sort_values(["model", "dataset"]).reset_index(drop=True))
print(f"Validated {len(metrics)} aggregate metric rows and all per-case records.")

,model,dataset,role,repo_id,config_name,split,requested_revision,resolved_revision,n_examples,n_directed_cases
0,gpt2-small,dynasent_r1,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r1_gpt2_small_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,80,80
1,gpt2-small,dynasent_r2,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r2_gpt2_small_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,66,66
2,gpt2-small,imdb,frozen_ood_evaluation,kokolamba/sentiment-manifold-imdb-pythia-2.8b,gpt2_small_directed_pairs,test,06586d20342aa46fa61525b1f7609ab065983e3e,06586d20342aa46fa61525b1f7609ab065983e3e,15274,15274
3,gpt2-small,sst,frozen_ood_evaluation,kokolamba/sentiment-manifold-sst-pythia-2.8b,tigges_gpt2_small_directed_pairs,test,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,304,304
4,qwen-0.6b,dynasent_r1,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r1_qwen_0_6b_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,80,80
5,qwen-0.6b,dynasent_r2,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r2_qwen_0_6b_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,68,68
6,qwen-0.6b,imdb,frozen_ood_evaluation,kokolamba/sentiment-manifold-imdb-pythia-2.8b,qwen_0_6b_directed_pairs,test,06586d20342aa46fa61525b1f7609ab065983e3e,06586d20342aa46fa61525b1f7609ab065983e3e,15218,15218
7,qwen-0.6b,sst,frozen_ood_evaluation,kokolamba/sentiment-manifold-sst-pythia-2.8b,tigges_qwen_0_6b_directed_pairs,test,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,302,302


Validated 24 aggregate metric rows and all per-case records.


## 8. Display logit-flip and literal sign-flip tables

For every model and metric, rows are the three frozen direction methods and columns are the four evaluation datasets. Each cell is `Lxx (score%)`; the layer remains the ToyMovieReview ADVERB-selected layer shown above.

In [9]:
from IPython.display import Markdown

DATASET_LABELS = {
    "sst": "SST",
    "imdb": "IMDb",
    "dynasent_r1": "DynaSent R1",
    "dynasent_r2": "DynaSent R2",
}
TABLE_METRICS = {
    "logit_flip_percent": "Logit Flip Percent",
    "sign_flip_percent": "Literal Sign Flip Percent",
}

for model_name in ALL_MODEL_NAMES:
    model_metrics = metrics[metrics["model"] == model_name].copy()
    for metric_column, metric_title in TABLE_METRICS.items():
        if model_metrics[metric_column].isna().any():
            raise RuntimeError(f"Missing {metric_column} values for {model_name}.")
        display_values = model_metrics.copy()
        display_values["metric_value"] = pd.to_numeric(
            display_values[metric_column]
        )
        table = display_values.pivot(
            index="method", columns="dataset", values="metric_value"
        ).reindex(index=METHOD_ORDER, columns=DATASET_ORDER)
        table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
        table.columns = [DATASET_LABELS[dataset] for dataset in DATASET_ORDER]
        display(Markdown(f"### {MODEL_LABELS[model_name]} — {metric_title}"))
        display(
            table.style.format("{:.1f}").set_properties(**{"text-align": "center"})
        )

### GPT-2 Small — Logit Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,50.0,67.1,55.6,84.4
Logistic Regression,46.7,63.4,38.9,62.5
DAS (1D),61.7,67.2,105.6,71.9


### GPT-2 Small — Literal Sign Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,0.0,1.1,0.0,0.0
Logistic Regression,0.0,0.9,0.0,0.0
DAS (1D),0.0,1.2,0.0,0.0


### Qwen3-0.6B Base — Logit Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,82.4,78.3,64.1,66.2
Logistic Regression,79.9,76.3,64.1,72.1
DAS (1D),70.5,57.0,46.9,45.6


### Qwen3-0.6B Base — Literal Sign Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,69.9,71.7,50.0,64.7
Logistic Regression,66.9,69.7,50.0,61.8
DAS (1D),58.3,51.8,38.7,44.1


## 9. Final paths and credential cleanup

In [10]:
clear_hf_credentials(config.hf_token_env)
assert config.hf_token_env not in os.environ
assert "HF_TOKEN" not in _RUNTIME_SECRETS
assert "_token" not in globals()
print("Verified: HF_TOKEN is absent from the environment and notebook secret cache.")
print("Completed run: ", RUN_LAYOUT.root)
print("Manifest:      ", RUN_LAYOUT.manifest_path)
print("Combined CSVs: ", RUN_LAYOUT.results_dir)
print("Per-model CSVs:")
for model_name in ALL_MODEL_NAMES:
    print("  ", RUN_LAYOUT.results_dir / model_name)

Verified: HF_TOKEN is absent from the environment and notebook secret cache.
Completed run:  /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT
Manifest:       /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/run_manifest.json
Combined CSVs:  /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/results
Per-model CSVs:
   /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/results/gpt2-small
   /content/drive/MyDrive/sentiment-geometry/end-position-ood-evaluation/runs/2026-09-22_00-18_CDT/results/qwen-0.6b


In [11]:
#Delete the runtime secret
_RUNTIME_SECRETS.clear()

## Interpretation boundary

These results test cross-dataset transfer of directions and layers selected previously on ToyMovieReview ADVERB. They do not authorize layer, method, checkpoint, dataset subset, or intervention-strength reselection from SST, IMDb, or DynaSent results. Differences among datasets should be interpreted together with their model-specific directed-pair counts and preprocessing provenance.